In [1]:
import pandas as pd

In [2]:
# INPUTS
SEASON = '2024-2025'
LEAGUE_SLUG = 'Premier-League'

In [6]:
possession_split_path = f'../../data/features/context/possession_split_{SEASON}_{LEAGUE_SLUG}.parquet'

ps = pd.read_parquet(possession_split_path)



In [4]:
fixtures = pd.read_csv(f'../../data/raw/fbref/fixtures_{SEASON}_{LEAGUE_SLUG}.csv')
team_stats = pd.read_csv(f'../../data/raw/fbref/team_stats_{SEASON}_{LEAGUE_SLUG}.csv')

In [5]:
team_stats[['match_id','home_possession','away_possession']]


,match_id,home_possession,away_possession
0,cc5b4244,55,45
1,a1d0d529,38,62
2,34557647,23,77
3,71618ace,40,60
4,4efc72e4,53,47
...,...,...,...
375,464cbad6,52,48
376,3d22336e,47,53
377,36844e73,65,35
378,1ff370e8,63,37


In [9]:
ps['exp_pos_home'].describe()

count    370.000000
mean       0.501334
std        0.050172
min        0.357600
25%        0.467381
50%        0.502600
75%        0.535100
max        0.650100
Name: exp_pos_home, dtype: float64

In [ ]:
df = fixtures.merge(team_stats[['match_id','home_possession','away_possession']], on="match_id", how="inner")
df = df.merge(ps[["match_id","exp_pos_home","exp_pos_away"]], on="match_id", how="inner")
# df["shots_total"] = df["home_shots"] + df["away_shots"]

df[["exp_pos_home","home_possession"]].corr()           # correlation
# (df["shots_total"] - df["exp_pace_total"]).describe() # residuals


,exp_pos_home,home_possession
exp_pos_home,1.000000,0.619274
home_possession,0.619274,1.000000


In [11]:
bins = pd.qcut(df["exp_pos_home"], 10, duplicates="drop")
df.groupby(bins)["home_possession"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pos_home"].mean())


/var/folders/l_/pcb32qsn4vz5n55rmw_4dctc0000gn/T/ipykernel_20036/3691215580.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(bins)["home_possession"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pos_home"].mean())


,obs,pred
exp_pos_home,,
"(0.357, 0.435]",38.789474,0.413729
"(0.435, 0.458]",45.444444,0.450236
"(0.458, 0.476]",45.473684,0.467897
"(0.476, 0.49]",48.333333,0.483064
"(0.49, 0.503]",50.457143,0.496501
"(0.503, 0.51]",50.472222,0.506472
"(0.51, 0.528]",54.648649,0.519296
"(0.528, 0.541]",57.648649,0.534695
"(0.541, 0.567]",60.621622,0.553811
